In [12]:
import numpy as np
import tensorflow_datasets as tfds
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd

In [13]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Spam Email/updated spam detector dataset.csv', encoding='latin-1')

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    df['Message'], df['Category'],
    test_size=0.2,
    random_state=42,
    stratify=df['Category']
)

In [15]:
VOCAB_SIZE = 10000
MAX_LEN = 100
BATCH_SIZE = 32
BUFFER_SIZE = 10000

In [16]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorizer.adapt(x_train.values)

In [17]:
train_dataset = tf.data.Dataset.from_tensor_slices((x_train.values, y_train.values)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test.values, y_test.values)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [18]:
model = tf.keras.Sequential([
    vectorizer,
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=64, mask_zero=True),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [19]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=['accuracy']
)

In [20]:
# Train model
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset
)

Epoch 1/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 31s 92ms/step - accuracy: 0.7815 - loss: 0.6118 - val_accuracy: 0.8992 - val_loss: 0.3543
Epoch 2/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 40s 90ms/step - accuracy: 0.9170 - loss: 0.2943 - val_accuracy: 0.9498 - val_loss: 0.1860
Epoch 3/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 42s 92ms/step - accuracy: 0.9693 - loss: 0.1410 - val_accuracy: 0.9683 - val_loss: 0.1160
Epoch 4/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 27s 91ms/step - accuracy: 0.9828 - loss: 0.0772 - val_accuracy: 0.9695 - val_loss: 0.0960
Epoch 5/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 26s 90ms/step - accuracy: 0.9899 - loss: 0.0493 - val_accuracy: 0.9717 - val_loss: 0.0860
Epoch 6/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 26s 90ms/step - accuracy: 0.9930 - loss: 0.0340 - val_accuracy: 0.9713 - val_loss: 0.1555
Epoch 7/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 26s 90ms/step - accuracy: 0.9924 - loss: 0.0328 - val_accuracy: 0.9764 - val_loss: 0.0806
Epoch 8/10
292/292 ━━━━━━━━━━━━━━━━━━━━ 28s 97ms/step - accuracy: 0.9959 - loss: 0.0202 - 

In [23]:
new_emails = [
    "As a valued customer, I am pleased to advise you that following recent review of your Mob No. you are awarded with a £1500 Bonus Prize, call 09066364589"
]

new_emails_dataset = tf.data.Dataset.from_tensor_slices(new_emails).batch(1)

predictions = model.predict(new_emails_dataset)

if predictions[0] > 0.5:
    print("Spam")
else:
    print("Not Spam")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step
Spam


In [24]:
model.save("spam_email.detector.keras")